# 05 — Quantum Teleportation

Quantum teleportation transmits an unknown qubit state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ from Alice to Bob using:
1. A shared entangled Bell pair $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$.
2. Two classical bits of communication.

In this notebook, we verify the protocol step by step using `quantum_ed`.

In [ ]:
import numpy as np

from quantum_ed.states import ket0, ket1, normalize
from quantum_ed.gates import H, X, Z, CNOT, kron_n, apply
from quantum_ed.linalg import dagger
from quantum_ed.density import rho_from_ket, fidelity_pure

## 1. Prepare Alice's input state $|\psi\rangle$

We choose an arbitrary single-qubit state:
$$|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$$

In [ ]:
theta = 0.6
phi = 1.2
psi = np.array([[np.cos(theta / 2)], [np.exp(1j * phi) * np.sin(theta / 2)]], dtype=complex)
psi = normalize(psi)

print("State |psi>:\n", psi)

## 2. Prepare the 3-qubit initial state

- Qubit 0: Alice's state $|\psi\rangle$
- Qubits 1 and 2: Shared Bell pair $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$

Qubit 1 belongs to Alice, and Qubit 2 belongs to Bob.

In [ ]:
I2 = np.eye(2, dtype=complex)

# Bell pair |Phi+> on qubits 1 and 2
bell_pair = apply(CNOT, apply(kron_n(H, I2), kron_n(ket0(), ket0())))

# 3-qubit composite state: |psi> (x) |Phi+>
state = kron_n(psi, bell_pair)

## 3. Alice's Bell-basis measurement operations

Alice couples her qubit to her half of the Bell pair:
1. $\text{CNOT}$ with control = qubit 0, target = qubit 1.
2. $H$ on qubit 0.

In [ ]:
# CNOT on qubits 0 and 1, identity on qubit 2
cnot_01 = kron_n(CNOT, I2)
state = apply(cnot_01, state)

# H on qubit 0, identity on qubits 1 and 2
h_0 = kron_n(H, I2, I2)
state = apply(h_0, state)

## 4. Measurement and Classical Feedforward Correction

Alice measures qubits 0 and 1 in the computational basis. There are 4 possible measurement outcomes $(m_0, m_1) \in \{0, 1\}^2$, each occurring with probability $1/4$.

Depending on $(m_0, m_1)$, Bob applies a correction gate $X^{m_1} Z^{m_0}$ to his qubit 2:
- $(0, 0) \implies I$ (state is already $|\psi\rangle$)
- $(0, 1) \implies X$ (bit flip)
- $(1, 0) \implies Z$ (phase flip)
- $(1, 1) \implies X Z$ (bit and phase flip)

Let's verify that for every possible measurement outcome, Bob recovers $|\psi\rangle$ with fidelity 1.0!

In [ ]:
proj = {
    0: ket0() @ dagger(ket0()),
    1: ket1() @ dagger(ket1())
}

for m0 in (0, 1):
    for m1 in (0, 1):
        # Projector for Alice's outcome on qubits 0 and 1
        P = kron_n(proj[m0], proj[m1], I2)
        unnorm_state = P @ state
        prob = float(np.real((dagger(unnorm_state) @ unnorm_state).item()))
        
        # Normalized collapsed state
        post_state = unnorm_state / np.sqrt(prob)
        
        # Extract Bob's qubit 2 state
        bob_state = np.zeros((2, 1), dtype=complex)
        for b in (0, 1):
            idx = (m0 << 2) | (m1 << 1) | b
            bob_state[b, 0] = post_state[idx, 0]
        bob_state = normalize(bob_state)
        
        # Bob applies feedforward correction: X^m1 @ Z^m0
        correction = np.eye(2, dtype=complex)
        if m0 == 1:
            correction = Z @ correction
        if m1 == 1:
            correction = X @ correction
        
        recovered = apply(correction, bob_state)
        fid = fidelity_pure(psi, rho_from_ket(recovered))
        print(f"Outcome (m0={m0}, m1={m1}): Prob = {prob:.4f}, Bob Fidelity = {fid:.4f}")
        assert np.isclose(fid, 1.0)

print("\nAll outcomes successfully recovered |psi> with unit fidelity!")

## Exercises

1) Verify what happens to Bob's qubit if Alice does not send her measurement results (i.e., trace out qubits 0 and 1 without correction). Confirm that Bob's reduced density matrix is maximally mixed $\frac{1}{2}I$, proving that teleportation does not allow faster-than-light communication (No-Signaling Theorem).
2) Teleport the eigenstates $|+\rangle$ and $|-\rangle$ and verify that the feedforward correction holds identically.